# Business Entity Resolution Pipeline (AWS SageMaker)
## ML Challenge 2026 - Cross-Source Entity Matching

This notebook provides an end-to-end, modular pipeline to be executed directly in an AWS SageMaker notebook instance.
It coordinates:
1. **Environment Setup & Dependency Check** (Linux/SageMaker packages & OpenMP)
2. **Path Verification & Exploratory Data Analysis** (Inspecting S1, S2, S3 & Ground Truth)
3. **Training & Candidate Generation** (Multi-key blocking + TF-IDF)
4. **Pairwise Feature Engineering & LightGBM Model Fitting**
5. **Threshold Sweeping** for Macro F0.5 optimization
6. **Inference on Test Set** (US, India, and France)
7. **Submission Verification** (TSV formatting, entity count & candidate constraints)
8. **Final Submission Package Creation** (Zip archive containing TSVs, code, and documentation)


### 1. Environment & Dependency Setup
Install required Python libraries and verify system packages. SageMaker instances typically use Amazon Linux 2 / AL2023.


In [ ]:
# Install Python dependencies from requirements.txt
!pip install -r code/business_entity_resolution/requirements.txt

# In SageMaker / Linux, ensure OpenMP (libgomp) is available for LightGBM
!command -v yum >/dev/null 2>&1 && sudo yum install -y libgomp || true
!command -v apt-get >/dev/null 2>&1 && sudo apt-get update && sudo apt-get install -y libgomp1 || true


In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import rapidfuzz
import unidecode
import joblib

# Add project root and code directory to python path
ROOT_DIR = Path.cwd().resolve()
CODE_DIR = ROOT_DIR / "code" / "business_entity_resolution"
sys.path.insert(0, str(CODE_DIR))

print(f"Working Directory: {ROOT_DIR}")
print(f"Code Directory: {CODE_DIR}")
print(f"LightGBM Version: {lgb.__version__}")
print(f"RapidFuzz Version: {rapidfuzz.__version__}")
print("Environment initialized successfully.")


### 2. Path Configuration & Data Verification
Verify directory layout and presence of training & test TSV files.


In [ ]:
TRAIN_DIR = ROOT_DIR / "dataset" / "train"
TEST_DIR = ROOT_DIR / "dataset" / "test"
OUTPUT_DIR = ROOT_DIR / "output"
ARTIFACTS_DIR = CODE_DIR / "artifacts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Checking Train Files:")
for f in ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv", "train_ground_truth.tsv"]:
    p = TRAIN_DIR / f
    status = f"FOUND ({p.stat().st_size / (1024*1024):.1f} MB)" if p.exists() else "MISSING"
    print(f"  {f:25s}: {status}")

print("\nChecking Test Files:")
for f in ["test_source1.tsv", "test_source2.tsv", "test_source3.tsv"]:
    p = TEST_DIR / f
    status = f"FOUND ({p.stat().st_size / (1024*1024):.1f} MB)" if p.exists() else "MISSING"
    print(f"  {f:25s}: {status}")


### 3. Exploratory Data Analysis & Quick Profiling
Inspect sample records, country distributions, and singleton rates across datasets.


In [ ]:
# Preview Source 1 train records
df_s1 = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", nrows=1000)
df_gt = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t", nrows=1000)

print(f"Source 1 Sample Count: {len(df_s1)}")
print(f"Countries in S1 sample: {df_s1['country'].value_counts().to_dict()}")

# Check singleton distribution in ground truth sample
gt_sample = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t", nrows=5000)
singletons = gt_sample['matched_entity_ids'].isna() | (gt_sample['matched_entity_ids'].str.strip() == '')
print(f"Singleton rate in sample: {singletons.mean():.2%}")

df_s1.head(3)


### 4. Training Pipeline: Blocking + Feature Extraction + LightGBM
Train the candidate generator and pairwise matcher.
The pipeline:
1. Normalizes names and addresses
2. Generates candidate pairs using multi-key blocking (postal, sorted tokens, soundex, prefixes) + TF-IDF
3. Extracts pairwise similarity features (Jaro-Winkler, Levenshtein, Jaccard, Token containment)
4. Evaluates a validation holdout split by S1 entity to prevent data leakage
5. Sweeps probability thresholds to maximize **macro F0.5**
6. Saves model bundle to `artifacts/matcher.joblib`


In [ ]:
# Execute the modular training pipeline
%cd {CODE_DIR}
!python src/train.py --train-dir {TRAIN_DIR}
%cd {ROOT_DIR}


### 5. Inspect Trained Model Artifact
Load the saved bundle from `artifacts/matcher.joblib` and view feature importance and selected threshold.


In [ ]:
bundle_path = ARTIFACTS_DIR / "matcher.joblib"
if bundle_path.exists():
    bundle = joblib.load(bundle_path)
    print("Trained Model Bundle Keys:", list(bundle.keys()))
    print(f"Selected F0.5 Threshold: {bundle.get('threshold')}")
    model = bundle.get('model')
    if model and hasattr(model, 'feature_importances_'):
        feature_names = bundle.get('feature_names', [f'feat_{i}' for i in range(len(model.feature_importances_))])
        imp_df = pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
        print("\nTop 10 Feature Importances:")
        print(imp_df.head(10).to_string(index=False))
else:
    print(f"Model bundle not found at {bundle_path}")


### 6. Run Inference on Test Dataset
Generate candidates and predict entity matches on `dataset/test/` (including US, India, and France).
Outputs:
- `output/candidate_pairs.tsv`
- `output/matching_results.tsv`


In [ ]:
%cd {CODE_DIR}
!python src/infer.py --data-dir {TEST_DIR} --prefix test --out-dir {OUTPUT_DIR}
%cd {ROOT_DIR}


### 7. Submission TSV Validation
Strictly validate the submission against competition formatting constraints:
- Matches are a subset of candidates
- Correct headers and column counts
- Valid entity IDs from test set
- Every test S1 entity is accounted for (including France)


In [ ]:
%cd {CODE_DIR}
!python utils/validate_submission.py \
    --matching {OUTPUT_DIR / "matching_results.tsv"} \
    --candidate {OUTPUT_DIR / "candidate_pairs.tsv"} \
    --test-dir {TEST_DIR}
%cd {ROOT_DIR}


### 8. Preview Final Outputs
Inspect generated match rows.


In [ ]:
print("--- matching_results.tsv preview ---")
!head -n 10 {OUTPUT_DIR / "matching_results.tsv"}

print("\n--- candidate_pairs.tsv preview ---")
!head -n 10 {OUTPUT_DIR / "candidate_pairs.tsv"}


### 9. Create Submission Archive
Create the final zip file adhering to competition submission requirements:
- `output/matching_results.tsv`
- `output/candidate_pairs.tsv`
- `code/business_entity_resolution/src/`
- `code/business_entity_resolution/README.md`
- `code/business_entity_resolution/requirements.txt`
- `Documentation_template.md`


In [ ]:
import zipfile

zip_path = ROOT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # 1. Output TSVs
    if (OUTPUT_DIR / "matching_results.tsv").exists():
        zipf.write(OUTPUT_DIR / "matching_results.tsv", arcname="output/matching_results.tsv")
    if (OUTPUT_DIR / "candidate_pairs.tsv").exists():
        zipf.write(OUTPUT_DIR / "candidate_pairs.tsv", arcname="output/candidate_pairs.tsv")
    
    # 2. Code directory
    for root, _, files in os.walk(CODE_DIR):
        for f in files:
            if f.endswith(('.py', '.md', '.txt')) and not '__pycache__' in root:
                fp = Path(root) / f
                rel_path = fp.relative_to(ROOT_DIR)
                zipf.write(fp, arcname=str(rel_path))
                
    # 3. Documentation
    doc_path = ROOT_DIR / "Documentation_template.md"
    if doc_path.exists():
        zipf.write(doc_path, arcname="Documentation_template.md")

print(f"Created submission package: {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)")
